# 01 · Accessing Meteosat SEVIRI data: from account to first scene

**Goal of this notebook:** download *one* SEVIRI Level 1.5 scene from the EUMETSAT Data Store, open it with `satpy`,
crop it to Central Europe and look at the three images that matter for night-time fog:

1. `IR_108`: the 10.8 µm brightness temperature (a "thermal photo" of the scene)
2. `IR_108 − IR_039`: the brightness temperature difference (BTD), the physical fog signal
3. the `night_fog` RGB that forecasters use

**Why start with one scene?** Every later step (cropping, labelling, training) depends on this one working.
Bulk downloads come only once the pipeline works end-to-end.

**Runtime:** a CPU runtime is enough here. Save your GPU quota for training (notebook/step 8).

## 0 · One-time setup (do this yourself, before running)

1. Create a free EUMETSAT account (EUMETSAT User Portal / EO Portal).
2. Get your **API consumer key + secret**: https://api.eumetsat.int/api-key (log in with that account).
3. In Colab, open the **🔑 Secrets** panel (left sidebar) and add two secrets:
   - `EUMETSAT_KEY` → your consumer key
   - `EUMETSAT_SECRET` → your consumer secret

   Switch on "Notebook access" for both.

**Why Colab Secrets?** The key never appears in the notebook, so you can push this notebook to a public GitHub
repo safely. That is the same rule as "never commit credentials", applied to notebooks.

## 1 · Install the libraries

In [ ]:
# eumdac  : EUMETSAT Data Access Client (search + download from the Data Store)
# satpy   : reads SEVIRI files, calibrates them, builds RGB composites
# pyresample / pyorbital : geolocation, cropping, sun angles (satpy dependencies)
# cartopy : maps with coastlines and borders
!pip install -q eumdac satpy pyresample pyorbital cartopy

In [ ]:
import datetime as dt
import shutil
import zipfile
from pathlib import Path

import eumdac
import matplotlib.pyplot as plt
import numpy as np
from google.colab import drive, userdata

print("eumdac", eumdac.__version__ if hasattr(eumdac, "__version__") else "")

## 2 · Where files go: Colab disk vs Google Drive

- `/content/...` is the Colab VM's own disk: fast and big, but **deleted when the runtime stops**.
  Raw full-disk files go here, because we only need them until we have cropped them.
- Google Drive is small but **persistent**. Only the small, processed outputs go there.

This is "strategy A" from the brief: download → crop → keep the small result → delete the raw file.

In [ ]:
drive.mount("/content/drive")

DRIVE_DIR = Path("/content/drive/MyDrive/fogseg")  # persistent, small outputs
RAW_DIR = Path("/content/raw")  # temporary, big raw files
FIG_DIR = DRIVE_DIR / "figures"
for d in (DRIVE_DIR, RAW_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

## 3 · Authenticate and open the collection

- An **access token** is a short-lived ticket that eumdac gets with your key + secret. It refreshes itself.
- The **Data Store** is EUMETSAT's online archive. It is organised in **collections**: one collection = one
  product type from one service.
- `EO:EUM:DAT:MSG:HRSEVIRI` = *High Rate SEVIRI Level 1.5 Image Data, MSG, 0° service*: the full-disk image
  from the operational Meteosat Second Generation satellite at 0° longitude, every 15 minutes.

In [ ]:
import requests

# .strip() removes invisible spaces/newlines that often sneak in when copy-pasting keys
key = (userdata.get("EUMETSAT_KEY") or "").strip()
secret = (userdata.get("EUMETSAT_SECRET") or "").strip()
print(f"Key length: {len(key)}, secret length: {len(secret)}")  # lengths only, never the values
assert key and secret, "One of the secrets is empty: check the names in the key panel"

token = eumdac.AccessToken((key, secret))
try:
    print("Token valid until:", token.expiration)
except requests.exceptions.HTTPError as err:
    # Show why the EUMETSAT server refused us (status code + its message), not the credentials
    print("Token request failed:", err.response.status_code, err.response.text[:300])
    raise

datastore = eumdac.DataStore(token)
collection = datastore.get_collection("EO:EUM:DAT:MSG:HRSEVIRI")
print(collection.title)
print()
print(collection.abstract[:800], "...")

**Processing levels, in one line each** (interview question!):

| Level | What it is |
|---|---|
| 0 | raw instrument telemetry, as downlinked |
| 1 / 1.5 | geolocated, calibrated radiances. For SEVIRI, 1.5 means *rectified* onto a fixed geostationary grid, so every pixel is the same place in every image |
| 2 | geophysical products derived from 1.5 (cloud mask, cloud top height, ...) |

We use Level 1.5 because we want to derive our own product (a fog mask) from the calibrated channels.

## 4 · Search for one night-time slot

SEVIRI scans the full disk every **15 minutes** (the *repeat cycle*). We search a 20-minute window around
midnight UTC, so we should get one or two products.

Any autumn/winter night works for this first test. We'll pick a famous fog night as the "hero" example later.

In [ ]:
start = dt.datetime(2023, 11, 20, 23, 50)
end = dt.datetime(2023, 11, 21, 0, 10)

products = list(collection.search(dtstart=start, dtend=end, sort="start,time,1"))
print(f"{len(products)} product(s) found")
for p in products:
    print(p, "| sensing", p.sensing_start, "->", p.sensing_end, "|", p.satellite, "| size", p.size)

Note the satellite name (e.g. MSG3 = Meteosat-10, MSG4 = Meteosat-11). The prime 0° satellite changes over the
years. That's fine, because the instrument and the grid are the same. We'll still store the satellite name per
scene as metadata.

## 5 · Download only the file we need

A Data Store *product* is a package with several files (the `.nat` image file plus metadata). We list them and
download only the `.nat` (native format) file.

In [ ]:
product = products[0]
entries = list(product.entries)
print("Files in this product:", entries)

nat_entries = [e for e in entries if e.endswith(".nat")]
if nat_entries:
    nat_path = RAW_DIR / Path(nat_entries[0]).name
    if not nat_path.exists():  # resumable: don't download twice
        with product.open(entry=nat_entries[0]) as fsrc, open(nat_path, "wb") as fdst:
            shutil.copyfileobj(fsrc, fdst)
else:
    # Fallback: download the whole product (a zip) and extract the .nat from it
    zip_path = RAW_DIR / f"{product}.zip"
    if not zip_path.exists():
        with product.open() as fsrc, open(zip_path, "wb") as fdst:
            shutil.copyfileobj(fsrc, fdst)
    with zipfile.ZipFile(zip_path) as zf:
        name = next(n for n in zf.namelist() if n.endswith(".nat"))
        zf.extract(name, RAW_DIR)
    nat_path = RAW_DIR / name

print(nat_path, f"{nat_path.stat().st_size / 1e6:.0f} MB")

## 5b · No API key yet? Use a manually downloaded file instead

**Skip sections 3–5** and use this cell instead if your API key isn't working yet:

1. On https://data.eumetsat.int open *High Rate SEVIRI Level 1.5 Image Data - MSG - 0 degree*, pick a night
   (e.g. 2023-11-20, 00:00 UTC) and download one product. You get a `.zip`.
2. In Google Drive, put the `.zip` (or the `.nat` inside it) into `MyDrive/fogseg/manual/`.
3. Run this cell. It finds the file and sets `nat_path`, just like section 5 would.

This is fine for exploring one scene. For the ~800 scenes of the real dataset we use the API (sections 3–5).

In [ ]:
MANUAL_DIR = DRIVE_DIR / "manual"
MANUAL_DIR.mkdir(exist_ok=True)

nats = sorted(MANUAL_DIR.glob("*.nat"))
zips = sorted(MANUAL_DIR.glob("*.zip"))
if nats:
    nat_path = nats[0]
elif zips:
    with zipfile.ZipFile(zips[0]) as zf:
        name = next(n for n in zf.namelist() if n.endswith(".nat"))
        zf.extract(name, RAW_DIR)  # extract to the fast temporary disk, not Drive
    nat_path = RAW_DIR / name
else:
    raise FileNotFoundError(f"Put a .zip or .nat file into {MANUAL_DIR} first")

print(nat_path, f"{nat_path.stat().st_size / 1e6:.0f} MB")

That one file is the **whole Earth disk, all 12 channels**, for one 15-minute slot. Multiply by ~800 scenes and
you see why we crop immediately and never keep the raw files.

## 6 · Open the scene with satpy

`satpy` knows the SEVIRI native format (reader `seviri_l1b_native`). It turns raw counts into physical units:
for the infrared channels that means **brightness temperature in Kelvin**: the temperature a perfect black body
would need to emit the radiance the satellite measured.

In [ ]:
from satpy import Scene

scn = Scene(filenames=[str(nat_path)], reader="seviri_l1b_native")
print(scn.available_dataset_names())

The 8 infrared channels are our model inputs later. The names give the central wavelength in µm:
`IR_039` = 3.9 µm, `IR_108` = 10.8 µm, and so on. (`VIS006`, `VIS008`, `IR_016` and `HRV` are solar channels,
useless at night.)

`upper_right_corner="NE"` flips the image so that north is up and east is right. SEVIRI's native orientation
is the other way round.

In [ ]:
IR_CHANNELS = ["IR_039", "WV_062", "WV_073", "IR_087", "IR_097", "IR_108", "IR_120", "IR_134"]

scn.load(IR_CHANNELS + ["night_fog"], upper_right_corner="NE")

bt108 = scn["IR_108"]
print(bt108.attrs["units"], bt108.attrs["calibration"], bt108.shape)
assert bt108.attrs["units"] == "K"
bt108

## 7 · Crop to Central Europe

We keep the **native geostationary grid**, only cropped. No resampling means no interpolation artefacts, and as
long as the bounding box stays the same, every scene gets exactly the same pixel grid (we'll assert that in
the pipeline).

Pixel size is ~3 km directly below the satellite (0°N, 0°E) but ~4–5 km over Central Europe, because the
satellite looks at Europe at a slant.

In [ ]:
BBOX = (2, 43, 20, 55)  # lon_min, lat_min, lon_max, lat_max

ceu = scn.crop(ll_bbox=BBOX)
area = ceu["IR_108"].attrs["area"]
print("Cropped shape:", ceu["IR_108"].shape)
print(area)

## 8 · Is it really night? (preview of step 4)

The 3.9 µm channel also measures reflected sunlight during the day, which destroys the fog signal. So we only
trust pixels where the sun is well below the horizon: **solar zenith angle (SZA) > ~93–95°** (90° = sun on the
horizon, plus a margin for twilight).

In [ ]:
from pyorbital.astronomy import sun_zenith_angle

lons, lats = area.get_lonlats()
t = ceu["IR_108"].attrs["start_time"]
sza = sun_zenith_angle(t, lons, lats)
print("Scene time:", t)
print(f"SZA range over region: {np.nanmin(sza):.1f}° – {np.nanmax(sza):.1f}°")
print(f"Night fraction (SZA > 95°): {np.mean(sza > 95):.1%}")

## 9 · Look at the data: IR_108, BTD and the night_fog RGB

**The physics:** fog and low stratus are made of small water droplets. Those droplets emit *less efficiently* at
3.9 µm than at 10.8 µm. So at night:

- over fog / low stratus: `BT(10.8) − BT(3.9)` is **clearly positive** (a few K)
- over clear land: close to zero

The `night_fog` RGB puts this into one image. In satpy it is built from
red = `IR_120 − IR_108`, green = `IR_108 − IR_039` (with a CO₂ correction on 3.9 µm), blue = `IR_108`.
Fog / low stratus appears in **aqua / light green-blue** tones.

In [ ]:
import cartopy.feature as cfeature

try:  # newer satpy
    from satpy.enhancements.enhancer import get_enhanced_image
except ImportError:  # older satpy
    from satpy.writers import get_enhanced_image

crs = area.to_cartopy_crs()
btd = ceu["IR_108"] - ceu["IR_039"]
rgb = get_enhanced_image(ceu["night_fog"]).data.transpose("y", "x", "bands").values[..., :3]

fig, axes = plt.subplots(1, 3, figsize=(20, 6), subplot_kw={"projection": crs})
panels = [
    (ceu["IR_108"].values, {"cmap": "gray_r"}, "IR 10.8 µm brightness temperature [K]"),
    (btd.values, {"cmap": "RdBu_r", "vmin": -6, "vmax": 6}, "BTD = BT10.8 − BT3.9 [K]"),
    (np.clip(rgb, 0, 1), {}, "night_fog RGB"),
]
for ax, (img, kw, title) in zip(axes, panels):
    im = ax.imshow(img, transform=crs, extent=crs.bounds, origin="upper", **kw)
    ax.coastlines(color="yellow", linewidth=0.6)
    ax.add_feature(cfeature.BORDERS, edgecolor="yellow", linewidth=0.4)
    ax.set_title(title)
    if kw:
        plt.colorbar(im, ax=ax, shrink=0.7)

fig.suptitle(f"SEVIRI {ceu['IR_108'].attrs['platform_name']}  {t:%Y-%m-%d %H:%M} UTC")
out_png = FIG_DIR / f"first_scene_{t:%Y%m%dT%H%M}.png"
fig.savefig(out_png, dpi=120, bbox_inches="tight")
print("Saved", out_png)

**What to look for:**

- Cold (bright in `gray_r`) high clouds in IR_108 vs warm, smooth, flat areas in valleys or basins.
- In the BTD panel, red patches (positive BTD) that line up with smooth, aqua areas in the night_fog RGB:
  candidate fog / low stratus. Typical places: Po Valley, Swiss Plateau, Danube basin, northern Germany.
- If the whole region is cloudy or clear, that's fine. This scene is just the plumbing test.

## 10 · Clean up

The raw file lives on the temporary disk and would vanish anyway, but in the bulk pipeline we delete it right
after cropping, so the disk never fills up.

In [ ]:
# nat_path.unlink()   # uncomment once you're done exploring this scene

## ✅ What you should be able to explain now

- What the EUMETSAT Data Store is, what a *collection* is, and what `EO:EUM:DAT:MSG:HRSEVIRI` contains.
- Level 0 / 1.5 / 2, and why we start from Level 1.5.
- The 15-minute repeat cycle and why one full-disk file is big.
- Brightness temperature, and why IR channels work at night.
- Why the native grid is kept (no resampling), and why pixels are ~4–5 km over Europe.
- Why 3.9 µm only works at night (reflected sunlight) → solar zenith angle mask.
- The BTD fog signal and how the night_fog RGB shows it.

**Next:** step 3 turns this into a repeatable function: scene → cropped, calibrated arrays + metadata →
one Zarr store on Drive.